# 配对仿真与策略结果分布

共享外生随机源，各策略依据自己的报价决定成交。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

显式逐事件更新报价、现金和库存。这个简化模型忽略队列和自身冲击。

In [ ]:
def simulate(kappa,directions,uniforms,innovations):
    mid=100.;q=0;cash=0.;max_inventory=0;fills=0
    for side,u,shock in zip(directions,uniforms,innovations):
        center=mid-kappa*q;bid=center-.5;ask=center+.5
        distance=(ask-mid) if side==1 else (mid-bid)
        probability=min(1.,.3*np.exp(-max(distance,0)))
        if u<probability:
            if side==1:cash+=ask;q-=1
            else:cash-=bid;q+=1
            fills+=1
        mid+=.15*side+.1*shock;max_inventory=max(max_inventory,abs(q))
    return cash+q*mid,max_inventory,fills
results=[]
for path in range(500):
    directions=rng.choice([-1,1],300);uniforms=rng.random(300);innovations=rng.normal(size=300)
    static=simulate(0,directions,uniforms,innovations);feedback=simulate(.15,directions,uniforms,innovations)
    results.append([*static,*feedback])
results=np.array(results);print('mean static PnL, max inventory, fills; feedback:',results.mean(axis=0))

标准误来自独立路径间的配对差，不来自单条路径的事件数。

In [ ]:
difference=results[:,3]-results[:,0];se=difference.std(ddof=1)/np.sqrt(len(difference))
print('difference mean, SE, approximate interval:',difference.mean(),se,difference.mean()+np.array([-1,1])*1.96*se)
plt.hist(difference,bins=30);plt.xlabel('feedback minus static PnL');plt.ylabel('paths');plt.show()

## 自己试一试

让外生价格对方向的响应从 0.15 改到 0.4，再比较结果。

## 反馈

这改变逆向选择强度，会改变收益与库存分布。即使某策略仍领先，也只是这个生成模型内的结果。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。